# CLIP Embedding Walkthrough — Index, Query, Visualise

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ACFPeacekeeper/Image-Toolkit/main?filepath=docs/notebooks/clip_embedding_walkthrough.ipynb)

This notebook demonstrates the vector search layer of Image Toolkit:
1. Load CLIP and compute image embeddings for a small corpus
2. Run a text query and retrieve nearest neighbours
3. Visualise the embedding space with UMAP/PCA projection
4. Show how the Recommendation Engine (SQLite backend) stores and queries these embeddings

**Prerequisites**:
```bash
source .venv/bin/activate
pip install open-clip-torch umap-learn matplotlib scikit-learn
```

**GPU**: CLIP inference is GPU-optional. CPU works fine for < 500 images.

In [ ]:
import sys
import pathlib
import glob
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image

REPO_ROOT = pathlib.Path().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

# ── Corpus configuration ───────────────────────────────────────────────────────
# Point this at any directory of images. The walkthrough uses ASP test frames
# as a convenient small corpus (≈ 200 frames across 20 test sets).
CORPUS_GLOB = str(REPO_ROOT / 'data' / 'asp_test*' / '*.png')
MAX_IMAGES = 100   # limit for quick demo; raise to index more

all_paths = sorted(glob.glob(CORPUS_GLOB))[:MAX_IMAGES]
print(f'Corpus: {len(all_paths)} images from {CORPUS_GLOB}')

## 1. Load CLIP (ViT-B/32 via open-clip)

The Recommendation Engine uses OpenCLIP ViT-B/32 (laion400m_e32) for both
image and text embeddings. The 512-d embedding space is shared between modalities,
enabling cross-modal retrieval (text → image and image → image).

In [ ]:
# SKIP_CI  ← requires open-clip-torch download (~350 MB)
import open_clip

MODEL_NAME   = 'ViT-B-32'
PRETRAINED   = 'laion400m_e32'

model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=PRETRAINED, device=DEVICE
)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()

print(f'Model: {MODEL_NAME} ({PRETRAINED})')
print('Embedding dim: 512')

## 2. Compute image embeddings

In [ ]:
# SKIP_CI
from torch.utils.data import DataLoader, Dataset

class ImagePathDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.transform(img), i

dataset = ImagePathDataset(all_paths, preprocess)
loader  = DataLoader(dataset, batch_size=32, num_workers=0)

embeddings = np.zeros((len(all_paths), 512), dtype=np.float32)
with torch.no_grad():
    for imgs, idxs in loader:
        imgs = imgs.to(DEVICE)
        feats = model.encode_image(imgs)
        feats = feats / feats.norm(dim=-1, keepdim=True)   # L2-normalise
        embeddings[idxs.numpy()] = feats.cpu().numpy()
        print(f'  Encoded {idxs[-1].item() + 1}/{len(all_paths)} images...', end='\r')

print(f'\nEmbedding matrix: {embeddings.shape}')

## 3. Text query → nearest neighbours

In [ ]:
# SKIP_CI
TEXT_QUERIES = [
    "anime character with blue hair",
    "outdoor landscape with trees",
    "indoor scene with furniture",
]
TOP_K = 5

for query in TEXT_QUERIES:
    tokens = tokenizer([query]).to(DEVICE)
    with torch.no_grad():
        text_feat = model.encode_text(tokens)
        text_feat = (text_feat / text_feat.norm(dim=-1, keepdim=True)).cpu().numpy()[0]

    scores = embeddings @ text_feat          # cosine similarity (both L2-normed)
    top_k_idx = np.argsort(scores)[::-1][:TOP_K]

    print(f'\nQuery: "{query}"')
    fig, axes = plt.subplots(1, TOP_K, figsize=(3 * TOP_K, 3))
    for ax, idx in zip(axes, top_k_idx):
        img = Image.open(all_paths[idx]).convert('RGB')
        ax.imshow(img)
        ax.set_title(f'{scores[idx]:.3f}\n{pathlib.Path(all_paths[idx]).stem[:12]}', fontsize=8)
        ax.axis('off')
    fig.suptitle(f'Top-{TOP_K} for: "{query}"', fontsize=11)
    plt.tight_layout()
    plt.show()

## 4. Embedding space visualisation (PCA → 2D)

Colour by source dataset (which `asp_testN` directory the frame comes from).

In [ ]:
# SKIP_CI
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(embeddings)
print(f'PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}')

# Label by test dataset
labels = [pathlib.Path(p).parent.name for p in all_paths]
unique_labels = sorted(set(labels))
cmap = plt.get_cmap('tab20', len(unique_labels))
label_to_color = {l: cmap(i) for i, l in enumerate(unique_labels)}
colors = [label_to_color[l] for l in labels]

fig, ax = plt.subplots(figsize=(10, 7))
for label in unique_labels:
    mask = [l == label for l in labels]
    xy = coords_2d[mask]
    ax.scatter(xy[:, 0], xy[:, 1], s=30, color=label_to_color[label],
               label=label, alpha=0.7, edgecolors='none')

ax.legend(fontsize=7, ncol=3, markerscale=1.5, bbox_to_anchor=(1, 1))
ax.set_title('CLIP embedding space (PCA 2D) — coloured by source dataset')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
plt.tight_layout()
plt.show()

## 5. Recommendation Engine — SQLite store

The standalone `Recommendation-Engine/` sub-project migrated from Qdrant to SQLite
(pure Python, no Docker). This cell shows how to insert the embeddings computed above
into the SQLite store and retrieve similar images.

In [ ]:
# CPU-only cell — safe for CI

REC_ENGINE_ROOT = REPO_ROOT / 'Recommendation-Engine'
if not REC_ENGINE_ROOT.exists():
    print('Recommendation-Engine/ not found — skipping SQLite demo.')
else:
    sys.path.insert(0, str(REC_ENGINE_ROOT))
    try:
        from src.store import SQLiteStore
        db_path = str(OUT_DIR / 'demo_rec_engine.db') if 'OUT_DIR' in dir() else ':memory:'
        store = SQLiteStore(db_path)

        # Upsert first 10 embeddings as demo
        for i, (path, emb) in enumerate(zip(all_paths[:10], embeddings[:10])):
            store.upsert(path, emb.tolist())

        # Query nearest to the first image
        query_vec = embeddings[0].tolist()
        results = store.search(query_vec, top_k=5)
        print(f'SQLiteStore: top-5 neighbours for {pathlib.Path(all_paths[0]).name}')
        for rank, (path, score) in enumerate(results, 1):
            print(f'  {rank}. {pathlib.Path(path).name:40s}  score={score:.4f}')
    except ImportError as e:
        print(f'SQLiteStore not importable: {e}')